# Experiments

## Setup: Import Libraries and Scripts

In [5]:
import pandas as pd
from IPython.display import display, HTML
import optimize_prompt as opt  # Full optimization script
import zero_shot_baseline as zsb  # Zero-shot baseline script
import pickle
import os
from datetime import datetime

# Style for better table display
pd.set_option('display.max_columns', None)
pd.set_option('display.expand_frame_repr', False)
pd.set_option('display.max_colwidth', None)  # Show full content in cells

# Path for saving experiment runs
RUNS_PICKLE_PATH = 'experiment_runs.pkl'

## Define Experiments

Add/edit experiments here. Each is a dict with:
- `'name'`: A label for the experiment.
- `'script'`: 'optimize' or 'zero_shot'.
- Other keys: Parameters for main() (e.g., generations, model_name).

In [ ]:
experiments = [
        {
        'name': 'Full Optimization - w META INSTRUCTION 3 & META TEMPLATE 3 & INSTR_STRATEGIES_ORIGINAL w/o early stopping',
        'script': 'optimize',
        'generations': 30,
        'pop_size': 6,
        'train_sample_size': 80,
        'test_sample_size': 500,
        'model_name': 'google/gemini-2.5-flash',
        'use_bandit_instr': True,
        'use_bandit_template': True,
        'statutory_context_enabled': True,
        'contract_context_enabled': True
    },
]

experiments_backlog = [

        {
        'name': 'Full Optimization - w META INSTRUCTION 3 & META TEMPLATE 3 & INSTR_STRATEGIES_ORIGINAL w/o early stopping - No Bandit',
        'script': 'optimize',
        'generations': 20,
        'pop_size': 4,
        'train_sample_size': 10,
        'test_sample_size': 300,
        'model_name': 'google/gemini-2.5-flash-lite-preview-06-17',
        'use_bandit_instr': False,
        'use_bandit_template': False,
        'statutory_context_enabled': True,
        'contract_context_enabled': True
    },
        {
        'name': 'Full Optimization - w META INSTRUCTION 3 & META TEMPLATE 3 - No Bandit',
        'script': 'optimize',
        'generations': 40,
        'pop_size': 4,
        'train_sample_size': 10,
        'test_sample_size': 300,
        'model_name': 'google/gemini-2.5-flash-lite-preview-06-17',
        'use_bandit_instr': False,
        'use_bandit_template': False,
        'statutory_context_enabled': True,
        'contract_context_enabled': True
    },
]

## Run Experiments

This cell runs each experiment and collects results, saving them to a pickle file with a timecode.

In [ ]:
results = []

for exp in experiments:
    print(f"\n=== Running Experiment: {exp['name']} ===")
    script = exp.pop('script')  # Remove script key for passing to main
    name = exp.pop('name')  # Remove name for passing to main
    run_time = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    
    try:
        if script == 'optimize':
            result = opt.main(**exp)
        elif script == 'zero_shot':
            result = zsb.main(**exp)
        else:
            raise ValueError(f"Unknown script: {script}")
        
        # Flatten metrics for table
        metrics = result['test_metrics']
        flat_result = {
            'Experiment Name': name,
            'Script': script,
            **exp,  # Add back parameters
            'Best Instruction': result['best_instruction'],
            'Best Template': result['best_template'],
            'Sample Size': metrics['sample_size'],
            'Valid Predictions': metrics['valid_predictions'],
            'Total Predictions': metrics['total_predictions'],
            'Accuracy': metrics['accuracy'],
            'Precision': metrics['precision'],
            'Recall': metrics['recall'],
            'F1 Micro': metrics['f1_micro'],
            'F1 Macro': metrics['f1_macro'],
            'Adjusted F1 Macro': metrics['adjusted_f1_macro'],
            'Support (0/1)': f"{metrics['support'].get('0', 0)} / {metrics['support'].get('1', 0)}",
            'Unique y_true': ', '.join(metrics['unique_y_true']),
            'Unique y_pred': ', '.join(metrics['unique_y_pred']),
            'Detailed Report': metrics['detailed_report_string'],  # Full string for details
            'Full Classification Report (Dict)': metrics['classification_report'],  # Raw dict if needed
            'Run Time': run_time
        }
        results.append(flat_result)
    except Exception as e:
        print(f"Error in experiment '{name}': {e}")
        results.append({'Experiment Name': name, 'Error': str(e), 'Run Time': run_time})

# Load previous runs if exists
if os.path.exists(RUNS_PICKLE_PATH):
    with open(RUNS_PICKLE_PATH, 'rb') as f:
        past_runs = pickle.load(f)
else:
    past_runs = []

# Add new results to past runs and save
all_runs = past_runs + results
with open(RUNS_PICKLE_PATH, 'wb') as f:
    pickle.dump(all_runs, f)


=== Running Experiment: Full Optimization - w META INSTRUCTION 3 & META TEMPLATE 3 & INSTR_STRATEGIES_ORIGINAL w/o early stopping ===
============ Generation 1 ============


---- Sent in Batch 1 ----
Instruction: Classify the following clause from a Terms of Service contract as fair (0) or unfair (1) using the statutory context and contract context for better understanding. Respond only with '0' or '1'.
Clause: where these changes or suspensions would amount to a complete termination of the services you may be entitled to a refund of the reasonable part of any charges paid by you .
Statutory Context: According to art. 3 of the Directive 93/13 on Unfair Terms in Consumer Contracts, a contractual term is unfair if: 1) it has not been individually negotiated; and 2) contrary to the requirement of good faith, it causes a significant imbalance in the parties' rights and obligations, to the detriment of the consumer. This general definition is further specified in the Annex to the Directive, containing an indicative and non-exhaustive list of the terms which may be regarded as unfair, as well as in a few dozen judgments of the Court of Justice of the EU (Micklit

⭐ Adjusted F1 Macro Score: 0.7987
---- Sent in Batch 1 ----
Instruction: Classify the following clause from a Terms of Service contract as fair (0) or unfair (1) using the statutory context and contract context for better understanding. Respond only with '0' or '1'.
Clause: -lrb- evernote corporation , evernote gmbh and evernote brasil , as applicable , may be referred to in these terms of service as `` evernote , '' `` we '' and sometimes `` us '' -rrb- .
Statutory Context: According to art. 3 of the Directive 93/13 on Unfair Terms in Consumer Contracts, a contractual term is unfair if: 1) it has not been individually negotiated; and 2) contrary to the requirement of good faith, it causes a significant imbalance in the parties' rights and obligations, to the detriment of the consumer. This general definition is further specified in the Annex to the Directive, containing an indicative and non-exhaustive list of the terms which may be regarded as unfair, as well as in a few dozen judgme

⭐ Adjusted F1 Macro Score: 0.7971
---- Sent in Batch 1 ----
Instruction: Classify the following clause from a Terms of Service contract as fair (0) or unfair (1) using the statutory context and contract context for better understanding. Respond only with '0' or '1'.
Clause: if you choose , you may contribute website themes -lrb- `` custom themes '' -rrb- to the service for use by other users .
Statutory Context: According to art. 3 of the Directive 93/13 on Unfair Terms in Consumer Contracts, a contractual term is unfair if: 1) it has not been individually negotiated; and 2) contrary to the requirement of good faith, it causes a significant imbalance in the parties' rights and obligations, to the detriment of the consumer. This general definition is further specified in the Annex to the Directive, containing an indicative and non-exhaustive list of the terms which may be regarded as unfair, as well as in a few dozen judgments of the Court of Justice of the EU (Micklitz and Reich 2014).

⭐ Adjusted F1 Macro Score: 0.8098
---- Sent in Batch 1 ----
Instruction: Classify the following clause from a Terms of Service contract as fair (0) or unfair (1) using the statutory context and contract context for better understanding. Respond only with '0' or '1'.
Clause: it also applies even if ea knew or should have known about the possibility of such damage .
Statutory Context: According to art. 3 of the Directive 93/13 on Unfair Terms in Consumer Contracts, a contractual term is unfair if: 1) it has not been individually negotiated; and 2) contrary to the requirement of good faith, it causes a significant imbalance in the parties' rights and obligations, to the detriment of the consumer. This general definition is further specified in the Annex to the Directive, containing an indicative and non-exhaustive list of the terms which may be regarded as unfair, as well as in a few dozen judgments of the Court of Justice of the EU (Micklitz and Reich 2014). Examples of unfair clauses en

⭐ Adjusted F1 Macro Score: 0.8789
---- Sent in Batch 1 ----
Instruction: Classify the following clause from a Terms of Service contract as fair (0) or unfair (1) using the statutory context and contract context for better understanding. Respond only with '0' or '1'.
Clause: prices indicated for special cases , such as single rooms or discounts for children are provided as a guide only .
Statutory Context: According to art. 3 of the Directive 93/13 on Unfair Terms in Consumer Contracts, a contractual term is unfair if: 1) it has not been individually negotiated; and 2) contrary to the requirement of good faith, it causes a significant imbalance in the parties' rights and obligations, to the detriment of the consumer. This general definition is further specified in the Annex to the Directive, containing an indicative and non-exhaustive list of the terms which may be regarded as unfair, as well as in a few dozen judgments of the Court of Justice of the EU (Micklitz and Reich 2014). Exampl

⭐ Adjusted F1 Macro Score: 0.7694
---- Sent in Batch 1 ----
Instruction: Classify the following clause from a Terms of Service contract as fair (0) or unfair (1) using the statutory context and contract context for better understanding. Respond only with '0' or '1'.
Clause: however , if nintendo becomes aware of possibly unlawful or inappropriate user-generated content , ncl reserves the right to delete or to block access to such user-generated content at its own discretion .
Statutory Context: According to art. 3 of the Directive 93/13 on Unfair Terms in Consumer Contracts, a contractual term is unfair if: 1) it has not been individually negotiated; and 2) contrary to the requirement of good faith, it causes a significant imbalance in the parties' rights and obligations, to the detriment of the consumer. This general definition is further specified in the Annex to the Directive, containing an indicative and non-exhaustive list of the terms which may be regarded as unfair, as well as i

## Display Results Table

Interactive table with all parameters and metrics. Sorted by most recent run.

In [15]:
# Load all runs from pickle and display sorted by most recent run time
import pickle
import pandas as pd
from IPython.display import display, HTML

RUNS_PICKLE_PATH = 'experiment_runs.pkl'

if os.path.exists(RUNS_PICKLE_PATH):
    with open(RUNS_PICKLE_PATH, 'rb') as f:
        all_runs = pickle.load(f)
    # Sort by 'Run Time' descending
    all_runs_sorted = sorted(all_runs, key=lambda x: x.get('Run Time', ''), reverse=True)
    df_results = pd.DataFrame(all_runs_sorted)
    if not df_results.empty:
        styled_df = df_results.style.set_properties(**{'text-align': 'left', 'white-space': 'pre-wrap'}).set_table_styles([
            {'selector': 'th', 'props': [('text-align', 'left')]}
        ]).background_gradient(cmap='viridis', subset=['Adjusted F1 Macro'])
        display(HTML("<h3>All Experiment Runs (Most Recent First)</h3>"))
        display(styled_df)
    else:
        print("No results to display.")
else:
    print("No experiment runs found.")

,Experiment Name,Script,generations,pop_size,train_sample_size,test_sample_size,model_name,use_bandit_instr,use_bandit_template,statutory_context_enabled,contract_context_enabled,Best Instruction,Best Template,Sample Size,Valid Predictions,Total Predictions,Accuracy,Precision,Recall,F1 Micro,F1 Macro,Adjusted F1 Macro,Support (0/1),Unique y_true,Unique y_pred,Detailed Report,Full Classification Report (Dict),Run Time
0,Full Optimization - w META INSTRUCTION 3 & META TEMPLATE 3 & INSTR_STRATEGIES_ORIGINAL w/o early stopping,optimize,3.000000,2.000000,5.000000,50,google/gemini-2.5-flash,True,True,True,True,Read the question again carefully. Classify the following clause from a Terms of Service contract as fair (0) or unfair (1) using the statutory context and contract context for better understanding. Respond only with '0' or '1'.,Instruction: Statutory Context: Contract Context: Clause:,50,50,50,0.800000,0.200000,0.500000,0.800000,0.584718,0.584718,46.0 / 4.0,"1, 0","1, 0",precision recall f1-score support 0 0.9500 0.8261 0.8837 46 1 0.2000 0.5000 0.2857 4 accuracy 0.8000 50 macro avg 0.5750 0.6630 0.5847 50 weighted avg 0.8900 0.8000 0.8359 50,"{'0': {'precision': 0.95, 'recall': 0.8260869565217391, 'f1-score': 0.8837209302325582, 'support': 46.0}, '1': {'precision': 0.2, 'recall': 0.5, 'f1-score': 0.2857142857142857, 'support': 4.0}, 'accuracy': 0.8, 'macro avg': {'precision': 0.575, 'recall': 0.6630434782608696, 'f1-score': 0.584717607973422, 'support': 50.0}, 'weighted avg': {'precision': 0.8899999999999999, 'recall': 0.8, 'f1-score': 0.8358803986710964, 'support': 50.0}}",2025-07-23 00:46:11
1,Full Optimization - w META INSTRUCTION 3 & META TEMPLATE 3 & INSTR_STRATEGIES_ORIGINAL w/o early stopping,optimize,15.000000,4.000000,40.000000,300,google/gemini-2.5-flash-lite-preview-06-17,True,True,True,True,"Classify the provided Terms of Service clause. Output '0' if the clause is fair, and '1' if it is unfair.","**Task:** Classify the fairness of a legal clause. **Instructions:** **Context:** * **Statutory Context:** * **Contract Context:** **Clause to Evaluate:** --- --- **Classification:** (Respond with '0' for fair, '1' for unfair)",300,300,300,0.880000,0.409091,0.642857,0.880000,0.715909,0.715909,272.0 / 28.0,"1, 0","1, 0",precision recall f1-score support 0 0.9609 0.9044 0.9318 272 1 0.4091 0.6429 0.5000 28 accuracy 0.8800 300 macro avg 0.6850 0.7736 0.7159 300 weighted avg 0.9094 0.8800 0.8915 300,"{'0': {'precision': 0.9609375, 'recall': 0.9044117647058824, 'f1-score': 0.9318181818181818, 'support': 272.0}, '1': {'precision': 0.4090909090909091, 'recall': 0.6428571428571429, 'f1-score': 0.5, 'support': 28.0}, 'accuracy': 0.88, 'macro avg': {'precision': 0.6850142045454546, 'recall': 0.7736344537815126, 'f1-score': 0.7159090909090908, 'support': 300.0}, 'weighted avg': {'precision': 0.9094318181818182, 'recall': 0.88, 'f1-score': 0.8915151515151515, 'support': 300.0}}",2025-07-22 18:38:09
2,Full Optimization - w META INSTRUCTION 3 & META TEMPLATE 3 & INSTR_STRATEGIES_ORIGINAL w/o early stopping,optimize,20.000000,8.000000,20.000000,300,google/gemini-2.5-flash-lite-preview-06-17,True,True,True,True,"Read the question again carefully. Let's think step-by-step. First, understand the clause's core purpose. Then, evaluate if it grants disproportionate power or benefit to the service provider at the user's expense. Consider ambiguity or overly broad language that could be exploited. If the clause creates an unreasonable burden or limits recourse unduly, classify it as '1'. If it's balanced, reasonably protects user rights, is clear, specific, and demonstrably fair to both parties, classify it as '0'. Adhere to general principles of consumer protection and check for hidden disadvantages. The final output should be '0' if the clause is fair or '1' if it is unfair.","### **Legal Clause Fairness Assessment** This task requires you to classify the fairness of a specific legal clause within a Terms of Service contract. **STATUTORY CO